# Ch 1. 작은 모델의 부활

**SLM(Small Language Model)의 현재 좌표와 부활 동력 세 가지를 손으로 확인한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part1/ch01_return_of_slm.ipynb)

In [ ]:
# !pip install -q transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device 확인
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 최소 예제 — SmolLM2-135M 한 번 돌려보기

직접 만들기 전에 "이 정도 크기면 어떤 출력이 나오는지" 먼저 본다.
SmolLM2-135M을 Colab에서 띄우는 데 30초.

- 135M × 4byte ≈ **540MB** — 무료 Colab RAM(12GB)에 충분

**관찰할 것 3가지**:
1. 영어 동화는 자연스럽다 — TinyStories와 같은 도메인이라.
2. 한국어 프롬프트("옛날 옛적에")를 주면 토큰화는 되지만 문장이 깨진다.
3. 같은 프롬프트를 5번 돌려도 매번 다르다 — 샘플링이 확률적이기 때문.

In [ ]:
# hello_smollm.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

name = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)

prompt = "Once upon a time"
ids = tok(prompt, return_tensors="pt").input_ids
out = model.generate(ids, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.9)
print(tok.decode(out[0], skip_special_tokens=True))

In [ ]:
# 한국어 프롬프트로도 시도 — 어떻게 깨지는지 확인
prompt_ko = "옛날 옛적에"
ids_ko = tok(prompt_ko, return_tensors="pt").input_ids
print("토큰화 결과:", [tok.decode([t]) for t in ids_ko[0]])
out_ko = model.generate(ids_ko, max_new_tokens=30, do_sample=True, temperature=0.8)
print("출력:", tok.decode(out_ko[0], skip_special_tokens=True))

## 실전 — 세 크기 비교 (135M / 360M / 1.7B)

같은 프롬프트를 SmolLM2의 세 크기에 던져 어디서 "말이 되기 시작"하는지 본다.
Colab T4면 1.7B까지 RAM에 들어간다 (4byte × 1.7B ≈ 6.8GB).

**예상 결과**:
- **135M** — "the reason small language models came back is the reason small language models..." 같은 반복 자주 발생
- **360M** — 한 줄 정도는 그럴듯하지만 곧 주제 이탈
- **1.7B** — "...because of better data curation, distillation, and a focus on quality over quantity." 처럼 개념적으로 맞는 답이 나올 확률이 눈에 띄게 높아짐

**관찰 1**: 능력은 크기에 따라 계단식이 아니라 부드럽게 증가하지만, "말이 끊기지 않는" 임계점은 존재한다.

**관찰 2**: 우리가 만들 10M 모델은 이 비교 표에 끼지도 못한다. 하지만 도메인을 좁히면(TinyStories 동화만) 10M도 충분히 한 페이지 동화를 짓는다.

In [ ]:
# size_compare.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

prompt = "The reason small language models came back in 2024 is"
sizes = ["135M", "360M", "1.7B"]
for s in sizes:
    name = f"HuggingFaceTB/SmolLM2-{s}"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16)  # bfloat16으로 메모리 절반
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(
        ids, max_new_tokens=80,
        do_sample=False,  # Greedy — 비교를 위해 결정론적으로
        repetition_penalty=1.1,
    )
    print(f"\n=== {s} ===")
    print(tok.decode(out[0], skip_special_tokens=True))
    del model; torch.cuda.empty_cache()  # 다음 모델 로드 전 메모리 비우기

## 연습문제

1. SmolLM2-135M / 360M / 1.7B에 같은 한국어 프롬프트 ("옛날 옛적에 작은 마을에")를 줘봐라. 어디서 "한국어가 깨지는지" 토큰 단위로 관찰해보고 한 줄로 요약하라.
2. Phi-3-mini와 SmolLM2-1.7B는 비슷한 파라미터 (각 3.8B / 1.7B)인데 강·약점이 다르다. 두 모델의 학습 데이터 구성 차이를 (논문/블로그를 근거로) 한 문단으로 정리하라.
3. 이 챕터의 표 1 ("등급" 표)에 본인 노트북을 추가한다면 어디 칸에 적겠는가? 메모리·CPU/GPU 기준으로.
4. **(생각해볼 것)** 만약 distillation이 더 발전해서 1M 모델이 GPT-4의 80% 능력을 낸다면, 이 책의 어떤 챕터가 의미를 잃고 어떤 챕터는 더 중요해질까?

In [ ]:
# 연습문제 1 — 한국어 프롬프트로 세 크기 비교


In [ ]:
# 연습문제 2 — Phi-3-mini vs SmolLM2-1.7B 학습 데이터 비교 메모


In [ ]:
# 연습문제 3 — 본인 노트북 사양 확인
import platform
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"RAM: {ram_gb:.1f} GB")
except ImportError:
    print("psutil not installed — install with: pip install psutil")

print(f"OS: {platform.system()} {platform.machine()}")
if torch.cuda.is_available():
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No CUDA GPU detected")

In [ ]:
# 연습문제 4 — distillation 발전 시나리오 메모
